In [ ]:
!pip install prince

In [ ]:
import numpy as np
import pandas as pd
from scipy.stats import chi2_contingency
import matplotlib.pyplot as plt
from statsmodels.graphics.mosaicplot import mosaic
from sklearn.metrics.pairwise import cosine_similarity
import prince

## Cross-Restaurant Analysis

During the analysis of restaurant reviews for the hybrid ABSA system, I noticed that some restaurants exhibited remarkably similar sentiment patterns. This observation led to the hypothesis that restaurants with similar characteristics may also share similar customer complaints and strengths.

To investigate this, I selected four French restaurants at random with approximately the same overall rating. For each restaurant, I randomly sampled 200 customer reviews.

The reviews were then processed using the proposed hybrid **ABSA + LLM** pipeline to automatically identify aspects and assign sentiment labels. The resulting annotated dataset was used to compare aspect-level sentiment distributions across restaurants and evaluate whether similar restaurants exhibit comparable negative feedback patterns.

In [1]:
import pandas as pd
import kagglehub

path = kagglehub.dataset_download(
    "achilov15/french-res-processed/versions/2"
)

print("Path:", path)
df = pd.read_csv(f"{path}/classified_df.csv")

df.head()

Path: C:\Users\timur\.cache\kagglehub\datasets\achilov15\french-res-processed\versions\2


,Unnamed: 0,restaurant,id,text,stars,aspect,polarity,proba_predicted_label,confidence,proba_conflict,...,human_checked,corrected_aspect,corrected_polarity,keep,comment,threshold,timestamp,aspect_sentiment,aspect_polarity,aspect_category
0,0,Lilette,0,"We ate dinner here on Oct. 29, 2014. This res...",3.0,dinner,negative,negative,0.627231,0.063512,...,False,dinner,negative,1,NaN,0.6,2026-07-28T12:45:05.104258,dinner_negative,dinner_negative,specific_dish
1,1,Lilette,0,"We ate dinner here on Oct. 29, 2014. This res...",3.0,restaurant,negative,negative,0.664401,0.062805,...,False,restaurant,negative,1,NaN,0.6,2026-07-28T12:45:05.104278,restaurant_negative,restaurant_negative,restaurant_general
2,2,Lilette,0,"We ate dinner here on Oct. 29, 2014. This res...",3.0,restaurant,negative,negative,0.664401,0.062805,...,False,restaurant,negative,1,NaN,0.6,2026-07-28T12:45:05.104285,restaurant_negative,restaurant_negative,restaurant_general
3,3,Lilette,0,"We ate dinner here on Oct. 29, 2014. This res...",3.0,food,negative,negative,0.642056,0.063764,...,False,food,negative,1,NaN,0.6,2026-07-28T12:45:05.104291,food_negative,food_negative,food_quality
4,4,Lilette,0,"We ate dinner here on Oct. 29, 2014. This res...",3.0,atmosphere,negative,negative,0.633703,0.076215,...,False,atmosphere,negative,1,NaN,0.6,2026-07-28T12:45:05.104296,atmosphere_negative,atmosphere_negative,atmosphere


In [ ]:
df.columns

In [ ]:
df.head()

In [ ]:
negative_counts = (
    df[df["corrected_polarity"] == "negative"]
    .groupby("restaurant")
    .size()
    .sort_values(ascending=False)
)

print(negative_counts)

It is important to note that these values represent **negative aspect mentions rather than negative reviews**. A single review may contain multiple negative aspects (e.g., food, service, and price), allowing one review to contribute more than one negative observation.

The results indicate that **Piquant generated substantially more negative aspect mentions** than the other restaurants, while **Lilette had the fewest**. This suggests noticeable differences in the volume of customer complaints despite all restaurants having similar overall ratings.

In [ ]:
negative_per_review = (
    df[df["corrected_polarity"] == "negative"]
    .groupby("restaurant")
    .size()
    /
    200
)

print(negative_per_review)

Among the four restaurants, **Piquant exhibited the highest density of negative aspect mentions**, averaging more than four negative aspects per review. In contrast, **Lilette showed the lowest density**, with approximately two negative aspects per review.

This finding suggests that, although the restaurants have similar overall ratings, the amount of detailed negative feedback varies considerably between them.

In [ ]:
df_test = df.copy()

In [ ]:
df_negative = df_test[
    df_test["corrected_polarity"]
    .astype(str)
    .str.lower()
    .eq("negative")
].copy()

In [ ]:
df_negative_unique = (
    df_negative
    .drop_duplicates(
        subset=["restaurant", "id", "aspect_category"]
    )
    .copy()
)

## Chi-Square Test of Independence

To determine whether different restaurants tend to receive different types of customer complaints, a **Chi-Square Test of Independence** (`chi2_contingency`) was performed.

The `chi2_contingency` function from `scipy.stats` analyzes a contingency table and tests whether two categorical variables are statistically independent.

In this experiment:

- **Variable 1:** Restaurant
- **Variable 2:** Aspect category of a negative mention

The null hypothesis (**H₀**) states that the distribution of negative aspect categories is **the same across all restaurants**, meaning there is no association between the restaurant and the type of complaint.

The alternative hypothesis (**H₁**) states that the distribution of negative aspect categories **differs between restaurants**, indicating that certain restaurants receive different kinds of complaints more frequently.

In addition to the Chi-square statistic and p-value, **Cramer's V** is calculated to measure the strength of the association:
- around **0.10** → weak association
- around **0.30** → moderate association
- around **0.50** or higher → strong association

In [ ]:
table = pd.crosstab(
    df_negative_unique["restaurant"],
    df_negative_unique["aspect_category"]
)

display(table)


chi2, p_value, dof, expected = chi2_contingency(table)

n = table.to_numpy().sum()
rows, columns = table.shape

cramers_v = np.sqrt(
    chi2 / (n * min(rows - 1, columns - 1))
)

print(f"Chi-square = {chi2:.3f}")
print(f"Degrees of freedom = {dof}")
print(f"p-value = {p_value:.6g}")
print(f"Cramer's V = {cramers_v:.3f}")

Since the p-value (0.136) is greater than the significance level of 0.05, we fail to reject the null hypothesis.

This suggests that there is **no statistically significant association** between the restaurant and the category of negative aspects. In other words, the four restaurants exhibit **similar patterns of customer complaints**.

Furthermore, Cramer's V (0.142) indicates only a **weak association**, providing additional evidence that the differences between restaurants are relatively small.

These findings support the initial hypothesis that restaurants with similar characteristics may also share similar negative feedback patterns.

## Checking the Assumptions of the Chi-Square Test

Before interpreting the results of the Chi-Square test, it is important to verify that its assumptions are satisfied.

One of the key assumptions is that the **expected frequency** in each cell of the contingency table should not be too small. The Chi-Square approximation is considered reliable when:

- no expected frequency is less than **1**;
- at least **80%** of the cells have an expected frequency of **5 or greater**.

The code below calculates the expected frequencies produced by the Chi-Square test and reports:

- the **minimum expected frequency**;
- the **percentage of cells** with an expected frequency below 5.

In [ ]:
expected_df = pd.DataFrame(
    expected,
    index=table.index,
    columns=table.columns
)

percent_below_5 = (expected_df < 5).to_numpy().mean() * 100
minimum_expected = expected_df.to_numpy().min()

print(f"Minimum expected frequency: {minimum_expected:.3f}")
print(
    "The proportion of cells with an expected frequency of < 5: "
    f"{percent_below_5:.1f}%"
)

### Assumption Check Results

The smallest expected frequency was **1.825**, which is greater than the minimum acceptable value of 1.

However, **20.0%** of the contingency table cells have expected frequencies below 5. This means that exactly **80%** of the cells meet the recommended threshold (expected frequency ≥ 5).

Therefore, the assumptions of the Chi-Square test are considered **acceptable**, although the expected counts are close to the commonly recommended limit. The test results can be interpreted with reasonable confidence.

In [ ]:
category_counts = df_negative_unique["aspect_category"].value_counts()

rare_categories = category_counts[
    category_counts < 20
].index

df_negative_unique["aspect_category_test"] = (
    df_negative_unique["aspect_category"]
    .where(
        ~df_negative_unique["aspect_category"].isin(rare_categories),
        "other_relevant"
    )
)

In [ ]:
rare_categories = [
    "cleanliness",
    "location_access",
    "reservation"
]

df_negative_unique["aspect_category_test"] = (
    df_negative_unique["aspect_category"]
    .replace({
        category: "other_operational"
        for category in rare_categories
    })
)

In [ ]:
table_reduced = pd.crosstab(
    df_negative_unique["restaurant"],
    df_negative_unique["aspect_category_test"]
)

chi2, p, dof, expected = chi2_contingency(table_reduced)

n = table_reduced.to_numpy().sum()
r, k = table_reduced.shape

cramers_v = np.sqrt(
    chi2 / (n * min(r - 1, k - 1))
)

expected_df = pd.DataFrame(
    expected,
    index=table_reduced.index,
    columns=table_reduced.columns
)

print(table_reduced)
print(f"\nChi-square = {chi2:.3f}")
print(f"Degrees of freedom = {dof}")
print(f"p-value = {p:.6f}")
print(f"Cramer's V = {cramers_v:.3f}")
print(
    "Minimum expected frequency:",
    expected_df.to_numpy().min()
)
print(
    "The proportion of cells with an expected frequency of < 5:",
    f"{(expected_df < 5).to_numpy().mean() * 100:.1f}%"
)

The contingency table contained the frequency of each negative aspect category across the four restaurants.

The Chi-Square test produced the following results:

- **χ² = 35.99**
- **Degrees of freedom = 36**
- **p-value = 0.469**
- **Cramer's V = 0.118**

Since the p-value (0.469) is substantially greater than the significance level of 0.05, we fail to reject the null hypothesis.

This indicates that there is **no statistically significant association** between the restaurant and the category of negative aspects. In other words, the distribution of customer complaints is statistically similar across the four restaurants.

Furthermore, **Cramer's V = 0.118** indicates only a **weak association**, suggesting that any observed differences are relatively small in magnitude.

## Relative Distribution of Negative Aspect Categories

While the Chi-Square test determines whether the distributions differ statistically, it is also useful to examine the **relative frequency** of each negative aspect category within each restaurant.

The table below shows the proportion of each negative aspect category after normalizing the counts within each restaurant. Each row sums to 1 (100%), allowing direct comparison of complaint profiles across restaurants.

In [ ]:
restaurant_proportions = table_reduced.div(
    table_reduced.sum(axis=1),
    axis=0
)

display(restaurant_proportions.round(3))

### Interpretation

The normalized distributions reveal that the complaint profiles are remarkably similar across the four restaurants.

Across all restaurants, the largest share of negative mentions is consistently associated with:

- **Food quality**
- **Restaurant general**
- **Service**
- **Specific dish**
- **Staff behavior**

Conversely, categories such as **desserts**, **waiting time**, **menu variety**, and **other operational issues** account for a relatively small proportion of complaints in every restaurant.

Although small differences are visible for example, Pacific Crepes has a slightly higher proportion of complaints related to food quality, while Meauxbar shows a somewhat larger proportion of complaints about specific dishes these variations are modest and consistent with the Chi-Square test, which found no statistically significant differences between restaurants (p = 0.469).

Overall, the normalized proportions provide additional visual evidence that restaurants with similar characteristics tend to exhibit comparable patterns of customer complaints.

In [ ]:
mean_profile = restaurant_proportions.mean(axis=0)

mean_profile = mean_profile.sort_values(ascending=False)

print("Typical complaints profile:")
display(mean_profile.to_frame("mean_share").round(3))

The average complaint profile indicates that the most common sources of negative feedback are:

1. **Specific dish** (13.7%)
2. **Food quality** (13.2%)
3. **Service** (12.7%)
4. **Restaurant general** (12.1%)
5. **Staff behavior** (10.9%)

Together, these five categories account for approximately **62.6%** of all negative aspect mentions across the four restaurants.

In contrast, complaints related to **desserts**, **waiting time**, **other operational issues**, and **atmosphere** represent a much smaller share of the overall negative feedback.

These results suggest that, among restaurants with similar characteristics, customer dissatisfaction is concentrated primarily around food quality, individual dishes, service, and staff interactions rather than operational or environmental factors.

## Consistency of Complaint Categories Across Restaurants

To assess how consistently each complaint category appears across restaurants, summary statistics were calculated for the normalized complaint profiles.

For each aspect category, the following statistics are reported:

- **Mean share** – the average proportion of negative mentions across all restaurants.
- **Minimum share** – the lowest observed proportion among the restaurants.
- **Maximum share** – the highest observed proportion among the restaurants.
- **Standard deviation** – the variation in the proportion of complaints between restaurants.

Categories with a **low standard deviation** occur at similar rates across restaurants, whereas categories with a **higher standard deviation** indicate greater variability between restaurants.

In [ ]:
profile_summary = pd.DataFrame({
    "mean_share": restaurant_proportions.mean(),
    "min_share": restaurant_proportions.min(),
    "max_share": restaurant_proportions.max(),
    "std_between_restaurants": restaurant_proportions.std(ddof=1)
}).sort_values("mean_share", ascending=False)

display(profile_summary.round(3))

## Leave-One-Restaurant-Out Validation

To evaluate whether the average complaint profile can generalize to an unseen restaurant, a leave-one-restaurant-out validation procedure was applied.

At each iteration:

1. One restaurant was excluded from the dataset.
2. The complaint profiles of the remaining three restaurants were averaged.
3. This average profile was treated as the predicted complaint distribution for the excluded restaurant.
4. The predicted and observed distributions were compared using:
   - **Mean Absolute Error (MAE)**
   - **Jensen–Shannon distance**

This procedure tests whether the complaint profile learned from similar restaurants can approximate the complaint distribution of another restaurant that was not used to construct the profile.

### Evaluation Metrics

**Mean Absolute Error (MAE)** measures the average absolute difference between the predicted and observed category proportions.

A lower MAE indicates that the predicted share of complaints in each category is close to the actual distribution.

**Jensen–Shannon distance** measures the overall difference between two probability distributions. Unlike a simple category-by-category error, it evaluates how similar the complete complaint profiles are.

The Jensen–Shannon distance ranges from:

- **0** — identical distributions;
- larger values — increasingly different distributions.

Therefore, lower values for both metrics indicate better agreement between the predicted and observed complaint profiles.

In [ ]:
profiles = restaurant_proportions.copy()
results = []

for restaurant in profiles.index:
    train = profiles.drop(index=restaurant)
    predicted = train.mean(axis=0)
    observed = profiles.loc[restaurant]

    # Mean Absolute Error between category proportions
    mae = np.mean(np.abs(observed - predicted))

    # Jensen–Shannon distance
    p = observed.to_numpy(dtype=float)
    q = predicted.to_numpy(dtype=float)

    p = p / p.sum()
    q = q / q.sum()

    m = 0.5 * (p + q)

    def kl_divergence(a, b):
        mask = a > 0
        return np.sum(a[mask] * np.log(a[mask] / b[mask]))

    js_divergence = 0.5 * (
        kl_divergence(p, m) +
        kl_divergence(q, m)
    )
    js_distance = np.sqrt(js_divergence)

    results.append({
        "excluded_restaurant": restaurant,
        "MAE": mae,
        "Jensen_Shannon_distance": js_distance
    })

validation = pd.DataFrame(results)

display(validation.round(3))
print("\nAverage MAE:", validation["MAE"].mean().round(3))
print(
    "Average Jensen–Shannon distance:",
    validation["Jensen_Shannon_distance"].mean().round(3)
)

### Results

The leave-one-restaurant-out validation produced the following average results:

- **Average MAE = 0.016**
- **Average Jensen–Shannon distance = 0.098**

The average MAE of 0.016 means that the predicted proportion for an aspect category differed from the observed proportion by approximately **1.6 percentage points**, on average.

The average Jensen–Shannon distance of 0.098 also indicates that the predicted and observed complaint distributions were relatively similar.

Among the four restaurants, **Piquant** and **Meauxbar** were predicted most accurately, with MAE values of 0.011 and 0.012. **Lilette** showed the largest category-level prediction error, while **Pacific Crepes** had the largest Jensen–Shannon distance.

Overall, the results suggest that the average complaint profile derived from three similar restaurants can provide a reasonably close approximation of the complaint distribution of a fourth restaurant.

## Visual Comparison of Complaint Profiles

To complement the statistical analysis, the normalized complaint profiles are visualized for each restaurant.

Each column represents one restaurant, while the height of each rectangle corresponds to the proportion of negative mentions assigned to a particular aspect category. Because the values are normalized, every column represents 100% of the negative complaint profile, allowing direct comparison between restaurants regardless of the total number of complaints.

In [ ]:
mosaic_data = {
    (restaurant, category): int(table_reduced.loc[restaurant, category])
    for restaurant in table_reduced.index
    for category in table_reduced.columns
}

fig, ax = plt.subplots(figsize=(20, 10))

mosaic(
    mosaic_data,
    ax=ax,
    gap=0.01,
    labelizer=lambda key: key[1].replace("_", " ")
)

ax.set_title(
    "Distribution of negative complaint categories across French restaurants",
    fontsize=16,
    pad=20
)

plt.tight_layout()
plt.show()

## Standardized Residual Analysis

Although the Chi-Square test indicates whether an overall association exists, it does not identify which individual cells contribute most to the test statistic.

To investigate this, **standardized residuals** were calculated for each combination of restaurant and complaint category.

A standardized residual measures how much the observed frequency differs from the expected frequency under the null hypothesis of independence.

The residual is calculated as:

\[
\text{Residual} = \frac{\text{Observed} - \text{Expected}}{\sqrt{\text{Expected}}}
\]

The interpretation is:

- **Positive residual** → the complaint category appears **more often than expected**.
- **Negative residual** → the complaint category appears **less often than expected**.
- Values close to **0** indicate that the observed frequency is close to the expected frequency.

As a rule of thumb:

- |Residual| < 2 → no meaningful deviation;
- |Residual| ≥ 2 → potentially important deviation.

In [ ]:
chi2, p, dof, expected = chi2_contingency(table_reduced)

expected_df = pd.DataFrame(
    expected,
    index=table_reduced.index,
    columns=table_reduced.columns
)

standardized_residuals = (
    table_reduced - expected_df
) / np.sqrt(expected_df)

display(standardized_residuals.round(2))

## The mosaic plot provides a graphical representation of the contingency table.

Each rectangle represents a combination of restaurant and complaint category:

- the width corresponds to the restaurant;
- the height corresponds to the proportion of complaints in each category.

The color of each rectangle is determined by the standardized residual:

- **Red** indicates that the category occurs more frequently than expected.
- **Blue** indicates that the category occurs less frequently than expected.
- **Grey** indicates that the observed frequency is close to the expected frequency.

In [ ]:
def residual_properties(key):
    restaurant, category = key
    residual = standardized_residuals.loc[restaurant, category]

    if residual > 2:
        # Observed count is noticeably higher than expected
        return {
            "facecolor": "red",
            "edgecolor": "white"
        }

    if residual < -2:
        # Observed count is noticeably lower than expected
        return {
            "facecolor": "blue",
            "edgecolor": "white"
        }

    # No substantial difference from the expected count
    return {
        "facecolor": "lightgray",
        "edgecolor": "white"
    }


mosaic_data = {
    (restaurant, category): int(table_reduced.loc[restaurant, category])
    for restaurant in table_reduced.index
    for category in table_reduced.columns
}

fig, ax = plt.subplots(figsize=(20, 10))

mosaic(
    mosaic_data,
    ax=ax,
    gap=0.01,
    properties=residual_properties,
    labelizer=lambda key: ""
)

ax.set_title(
    "Mosaic Plot of Negative Complaint Categories\n"
    "Red: more than expected; blue: fewer than expected",
    fontsize=16,
    pad=20
)

plt.tight_layout()
plt.show()

The mosaic plot is dominated by grey cells, indicating that most observed frequencies are close to their expected values.

Only one cell (Pacific Crepes – Atmosphere) shows a noticeable negative deviation, but even this deviation is relatively modest.

No strong clusters of red or blue cells are visible, suggesting that no restaurant consistently over- or under-represents particular complaint categories.

The mosaic plot therefore visually supports the statistical findings obtained from the Chi-Square test and standardized residual analysis, reinforcing the conclusion that the four restaurants share very similar complaint distributions.

## Correspondence Analysis

To further explore the relationship between restaurants and complaint categories, **Correspondence Analysis (CA)** was performed.

Correspondence Analysis is a multivariate exploratory technique designed for contingency tables. It projects both rows (restaurants) and columns (complaint categories) into a common low-dimensional space while preserving as much of the association structure as possible.

In the resulting plot:

- each **blue point** represents a restaurant;
- each **orange point** represents a complaint category.

Points that are located close to one another indicate a stronger association than would be expected under complete independence. Conversely, points near the origin represent average profiles with little deviation from the overall distribution.

It is important to note that Correspondence Analysis is primarily an exploratory visualization technique. It does not provide statistical significance tests but instead helps identify potential patterns that complement the Chi-Square analysis.

In [ ]:
# Correspondence Analysis
ca = prince.CA(
    n_components=2,
    n_iter=20,
    random_state=42
)

ca = ca.fit(table_reduced)

In [ ]:
row_coords = ca.row_coordinates(table_reduced)
col_coords = ca.column_coordinates(table_reduced)

display(row_coords)
display(col_coords)

### Principal Coordinates

The tables above report the coordinates of restaurants and complaint categories on the first two correspondence dimensions.

These coordinates define the positions shown in the correspondence map. Categories or restaurants with similar coordinates are positioned closer together, suggesting similar complaint profiles.

The correspondence map shows that all four restaurants are located relatively close to the center of the plot, indicating that none of them strongly deviates from the overall complaint distribution.

Most complaint categories are also concentrated around the origin, suggesting that they are shared across restaurants rather than being strongly associated with a particular establishment.

Some mild tendencies can be observed. For example:

- **Pacific Crepes** is positioned closer to **Food quality**.
- **Lilette** is located nearer **Seating comfort** and **Menu variety**.
- **Meauxbar** and **Piquant** remain close to the center, indicating complaint profiles that are very similar to the overall average.

However, these associations are relatively weak because the distances between restaurants are small and none of the restaurants is located far from the origin.

Overall, the correspondence analysis visually supports the findings of the Chi-Square test, standardized residual analysis, and normalized complaint profiles. The four restaurants exhibit highly similar complaint structures, with only minor differences in the relative emphasis of individual complaint categories.

In [ ]:
plt.figure(figsize=(10,8))

# restaurants
plt.scatter(
    row_coords[0],
    row_coords[1],
    s=150,
    marker='o'
)

for i, restaurant in enumerate(row_coords.index):
    plt.text(
        row_coords.iloc[i,0],
        row_coords.iloc[i,1],
        restaurant,
        fontsize=12,
        weight='bold'
    )

# categories
plt.scatter(
    col_coords[0],
    col_coords[1],
    s=100,
    marker='^'
)

for i, category in enumerate(col_coords.index):
    plt.text(
        col_coords.iloc[i,0],
        col_coords.iloc[i,1],
        category.replace("_"," "),
        fontsize=10
    )

plt.axhline(0,color='grey',linewidth=1)
plt.axvline(0,color='grey',linewidth=1)

plt.xlabel("Dimension 1")
plt.ylabel("Dimension 2")
plt.title("Correspondence Analysis of Complaint Categories")

plt.tight_layout()
plt.show()

In [ ]:
restaurant_profiles = table_reduced.div(
    table_reduced.sum(axis=1),
    axis=0
)

display(restaurant_profiles.round(3))

## Cosine Similarity of Complaint Profiles

To quantify the similarity between restaurants, cosine similarity was computed using the normalized complaint profiles.

Each restaurant is represented as a vector, where each element corresponds to the proportion of negative mentions assigned to a particular complaint category.

Cosine similarity measures the angle between two vectors rather than their magnitude. Since all complaint profiles are normalized, this metric evaluates how similar the overall distribution of complaint categories is between restaurants.

The cosine similarity ranges from:

- **1.0** – identical complaint profiles;
- **0** – completely unrelated profiles;
- **−1** – opposite profiles (not expected in this context).

Higher values therefore indicate more similar complaint distributions.

In [ ]:
cosine = cosine_similarity(restaurant_profiles)

cosine_df = pd.DataFrame(
    cosine,
    index=restaurant_profiles.index,
    columns=restaurant_profiles.index
)

display(cosine_df.round(4))

Overall, this study provides preliminary evidence that restaurants with similar characteristics tend to share comparable complaint profiles. Consequently, businesses planning to develop a restaurant with a similar concept should anticipate that many of the same customer concerns may arise, allowing proactive improvements before these issues become widespread.

### Limitations

Several limitations should be acknowledged.

- Only four restaurants were included in the analysis.
- Each restaurant contributed 200 randomly sampled reviews.
- All restaurants belonged to the same cuisine (French) and had similar overall ratings.

Therefore, the findings should be interpreted as evidence for this specific sample rather than as a general conclusion about all restaurants.

Future work should include a larger number of restaurants with different cuisines, locations, and rating levels to evaluate whether the observed complaint profile generalizes more broadly.